# Tiny Workflow Runner (Using OpenAI and DuckDB)

If running from within the piglets repository:

```bash
uv sync --extra examples --extra openai --extra duckdb
```

This example runs the currently available workflow stages: load a DuckDB search space, then generate a hypothesis with the logical planner.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
src_path = repo_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from piglets import (
    DatabaseConnector,
    DuckDBURL,
    GenerateHypothesis,
    LoadSearchSpace,
    LogicalPlanner,
    Question,
    WorkflowRunner,
    create_tpch_example_duckdb_db,
)

MODEL_NAME = "gpt-5.2"
DB_PATH = "data/tpch_sf1.duckdb"
QUESTION = """
Which manufacturers saw the largest increase in average revenue per order between 1996 and 1997,
considering only manufacturers with at least 100 orders in both years, and excluding cancelled orders?
"""

create_tpch_example_duckdb_db(db_path=DB_PATH)

question = Question(natural_language_question=QUESTION)
database_connector = DatabaseConnector(
    connection=DuckDBURL(database=DB_PATH),
)

runner = WorkflowRunner(
    stages=[
        LoadSearchSpace(database_connector),
        GenerateHypothesis(LogicalPlanner(MODEL_NAME, num_samples=3)),
    ]
)

context = runner.run(question)


In [ ]:
print("Database:", context.search_space.database_schema.name)
print("Tables:")
for table_schema in context.search_space.database_schema.table_schemas:
    print(f"- {table_schema.name}")

print("\nHypothesis technique:", context.hypothesis.technique)
print("Hypothesis parameters:", context.hypothesis.technique_parameters)
print("\nHypothesis:")
print(context.hypothesis.content)
